# 07 - PPMI Replication

Explore PPMI (Parkinson's) replication results from `data/results/ppmi/`.  
PD DNB results (primary), CSD results (secondary), cross-disease comparison.
**This notebook reads pre-computed results only.**

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib_venn import venn2
from scipy import stats

%matplotlib inline
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120

In [ ]:
with open('../config/config.yaml', 'r') as f:
    config = yaml.safe_load(f)

ppmi_dir = os.path.join('..', config['paths']['results'], 'ppmi')
adni_csd_dir = os.path.join('..', config['paths']['results'], 'csd')
adni_dnb_dir = os.path.join('..', config['paths']['results'], 'dnb')

print('PPMI results directory:', ppmi_dir)
print('Files available:', os.listdir(ppmi_dir) if os.path.exists(ppmi_dir) else 'DIRECTORY NOT FOUND')

## Figure 8A - PD CSD Volcano Plot

Kendall tau for variance: fast PD progressors vs. slow progressors.  
Side-by-side comparison with the ADNI AD result.

In [ ]:
# Load PPMI and ADNI CSD results
ppmi_tau = pd.read_csv(os.path.join(ppmi_dir, 'kendall_tau_variance.csv'))
adni_tau = pd.read_csv(os.path.join(adni_csd_dir, 'kendall_tau_variance.csv'))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, title, color in [(axes[0], adni_tau, 'ADNI (Alzheimer\'s)', 'firebrick'),
                              (axes[1], ppmi_tau, 'PPMI (Parkinson\'s)', 'darkgreen')]:
    neg_log_p = -np.log10(df['fdr_pvalue'].clip(lower=1e-300))
    sig_mask = df['fdr_pvalue'] < 0.05

    ax.scatter(df.loc[~sig_mask, 'effect_size'], neg_log_p[~sig_mask],
               c='grey', alpha=0.3, s=8)
    ax.scatter(df.loc[sig_mask, 'effect_size'], neg_log_p[sig_mask],
               c=color, alpha=0.6, s=12)
    ax.axhline(-np.log10(0.05), color='grey', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Effect Size (rank-biserial r)')
    ax.set_ylabel('-log10(FDR p-value)')
    ax.set_title(f'{title} (n={sig_mask.sum()} significant)')

plt.suptitle('CSD Variance Trend: Disease-Specific Volcano Plots', fontsize=13)
plt.tight_layout()
plt.show()

## Figure 8B - PD DNB Score Across Progression Groups

DNB scores for PD slow, intermediate, and fast progressors.

In [ ]:
# Load PPMI DNB scores
ppmi_dnb = pd.read_csv(os.path.join(ppmi_dir, 'dnb_scores_by_stage.csv'))

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(range(len(ppmi_dnb)), ppmi_dnb['mean_dnb_score'],
       yerr=ppmi_dnb['sem_dnb_score'], color='darkgreen', capsize=4, edgecolor='white')
ax.set_xticks(range(len(ppmi_dnb)))
ax.set_xticklabels(ppmi_dnb['stage'], rotation=15)
ax.set_ylabel('Mean DNB Score')
ax.set_title('PPMI: DNB Score Across PD Progression Groups')
plt.tight_layout()
plt.show()

print(ppmi_dnb[['stage', 'mean_dnb_score', 'sem_dnb_score', 'n']].to_string(index=False))

## Figure 9A - Cross-Disease Venn Diagram

Overlap of DNB core proteins between AD (ADNI) and PD (PPMI).  
Partial overlap is expected (different diseases, some shared neurodegeneration pathways).

In [ ]:
# Load core proteins from both cohorts
adni_core = pd.read_csv(os.path.join(adni_dnb_dir, 'dnb_core_proteins.csv'))
ppmi_core = pd.read_csv(os.path.join(ppmi_dir, 'dnb_core_proteins.csv'))

adni_set = set(adni_core['protein'])
ppmi_set = set(ppmi_core['protein'])
overlap = adni_set & ppmi_set

print(f'ADNI core proteins: {len(adni_set)}')
print(f'PPMI core proteins: {len(ppmi_set)}')
print(f'Shared proteins:    {len(overlap)}')

fig, ax = plt.subplots(figsize=(6, 5))
venn2([adni_set, ppmi_set], set_labels=('AD (ADNI)', 'PD (PPMI)'),
      set_colors=('firebrick', 'darkgreen'), alpha=0.5, ax=ax)
ax.set_title('DNB Core Protein Overlap: AD vs. PD')
plt.tight_layout()
plt.show()

if overlap:
    print(f'\nShared core proteins: {sorted(overlap)}')

## Figure 9B - Cross-Disease Pathway Enrichment Comparison

Side-by-side pathway enrichment for AD and PD DNB core proteins.

In [ ]:
# Load GSEA results for both cohorts
adni_gsea_path = os.path.join(adni_dnb_dir, 'gsea_results')
ppmi_gsea_path = os.path.join(ppmi_dir, 'gsea_results')

adni_gsea_files = [f for f in os.listdir(adni_gsea_path) if f.endswith('.csv')] if os.path.exists(adni_gsea_path) else []
ppmi_gsea_files = [f for f in os.listdir(ppmi_gsea_path) if f.endswith('.csv')] if os.path.exists(ppmi_gsea_path) else []

if adni_gsea_files and ppmi_gsea_files:
    adni_gsea = pd.read_csv(os.path.join(adni_gsea_path, adni_gsea_files[0]))
    ppmi_gsea = pd.read_csv(os.path.join(ppmi_gsea_path, ppmi_gsea_files[0]))

    # Merge on pathway
    merged_gsea = adni_gsea[['pathway', 'NES', 'padj']].merge(
        ppmi_gsea[['pathway', 'NES', 'padj']], on='pathway', suffixes=('_AD', '_PD'))

    # Filter to significant in at least one
    sig = merged_gsea[(merged_gsea['padj_AD'] < 0.25) | (merged_gsea['padj_PD'] < 0.25)]

    if len(sig) > 0:
        fig, ax = plt.subplots(figsize=(10, max(4, len(sig) * 0.5)))
        y_pos = range(len(sig))
        ax.barh([y - 0.15 for y in y_pos], sig['NES_AD'], height=0.3,
                color='firebrick', alpha=0.7, label='AD (ADNI)')
        ax.barh([y + 0.15 for y in y_pos], sig['NES_PD'], height=0.3,
                color='darkgreen', alpha=0.7, label='PD (PPMI)')
        ax.set_yticks(list(y_pos))
        ax.set_yticklabels(sig['pathway'], fontsize=8)
        ax.set_xlabel('Normalized Enrichment Score (NES)')
        ax.set_title('Cross-Disease Pathway Enrichment Comparison')
        ax.legend()
        plt.tight_layout()
        plt.show()
    else:
        print('No pathways significant in either cohort at FDR < 0.25.')
else:
    print('GSEA results not available for one or both cohorts.')